# 09. Hand Landmark - HandLandmarkLite_1

| 항목 | 내용 |
|---|---|
| 대상 모델 | `HandLandmarkLite_1.dxnn` |
| 카테고리 | **Hand Landmark** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 출력이 네 개일 때 각 tensor 의 역할을 이름 순서가 아닌 근거로 판별해 설명할 수 있다.
2. 좌표가 정규화값인지 입력 픽셀값인지 실제 값으로 판정하고 원본 좌표로 복원할 수 있다.
3. 21개 점의 번호와 손의 부위를 대응시켜 특정 손가락 끝의 좌표를 지목할 수 있다.
4. `z` 값이 절대 깊이가 아니라 손목 기준 상대값임을 실행 결과로 보일 수 있다.

## 1. Hand Landmark (손 관절점 검출)

**손이 찍힌 영상에서 정해진 개수의 관절 위치를 좌표로 찾아내는 작업**

손이 어디 있는지가 아니라 **손의 어느 부위가 어디 있는지**를 답한다. 점마다 번호가 정해져 있어 번호를 알면
그 점이 어느 관절인지 알 수 있고, 점들을 이어 손 모양을 그릴 수 있다.

**출력의 성격**

좌표 목록이며 점의 개수가 고정이다. 손이 없어도 좌표는 나오므로 **함께 나오는 신뢰도로 걸러야** 한다. 점의
순서가 곧 부위를 뜻하므로 순서가 어긋나면 손 모양이 엉킨다.

**활용 사례**

- 손짓 인식으로 기기 조작
- 수어 인식의 앞단
- 손 동작 기록·분석

**포함되지 않는 작업**

- 사진에서 손을 찾아 영역을 잡는 일 (Object Detection)
- 누구의 손인지 판정 (Face Embedding 계열)
- 손 영역을 픽셀 단위로 나누기 (Instance Segmentation)

## 2. 전체 파이프라인

```text
sample_hand.jpg  3009×2592 BGR
  → 전처리         BGR→RGB · 224×224 직접 축소 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → Public Output  Identity   [1, 63]  21점 × (x, y, z)
                   Identity_1 [1, 1]   손이 있을 확률
                   Identity_2 [1, 1]   왼손·오른손 점수
                   Identity_3 [1, 63]  후처리가 쓰지 않는 값
  → 후처리         21×3 으로 자르기 → 좌표 공간 판정 → 원본 크기로 복원 → 신뢰도로 걸러내기
  → Result         HandLandmarkResult (landmarks · confidence · handedness)
  → 시각화         21점과 연결선을 원본 위에 그림
```

![09. Hand Landmark 전체 파이프라인](resources/09_Hand_Landmark_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input_1` |
| Shape / Dtype | `[1, 224, 224, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**손이 화면을 채우고 있어야 한다**

- **손을 찾는 단계가 없다** — 사진 전체를 224×224 로 줄여 넣으므로 손이 작게 찍혀 있으면 관절을 구분할 화소가
  남지 않는다. 실제 제품에서는 손 검출을 앞에 붙여 잘라 넣는다
- **비율을 유지하지 않는다** — 3009×2592 가 224×224 로 눌리므로 가로가 더 많이 줄어든다. 좌표를 되돌릴 때
  축마다 다른 배율을 쓴다

### 3.2 출력

**출력이 네 개다.** 이 모델은 Task 가 `npu_0` 하나뿐이며 네 tensor 가 모두 밖으로 나온다.

| 런타임 순서 | Tensor | Shape / dtype | 크기 | 역할 |
|---:|---|---|---:|---|
| 0 | `Identity` | `[1, 63]` / `float32` | 252 B | 21점 × (x, y, z) |
| 1 | `Identity_1` | `[1, 1]` / `float32` | 4 B | 손이 있을 확률 |
| 2 | `Identity_2` | `[1, 1]` / `float32` | 4 B | 왼손·오른손 점수 |
| 3 | `Identity_3` | `[1, 63]` / `float32` | 252 B | 후처리가 쓰지 않는 값 |

네 개 모두 `InferenceEngine.run()` 이 돌려주는 Public Output 이다.

**이름과 순서만으로 역할을 단정할 수 없다.** `Identity`·`Identity_1` 같은 이름은 컴파일러가 붙인 것이고,
`parse_model` 이 나열하는 순서(`Identity` → `Identity_3` → `Identity_2` → `Identity_1`)와 런타임이 돌려주는
순서(`Identity` → `Identity_1` → `Identity_2` → `Identity_3`)도 다르다. **역할은 값을 보고 판정한다.**

| 판정 근거 | 관찰값 |
|---|---|
| 63 = 21 × 3 인 것 | `Identity` 와 `Identity_3` 둘 다 63개 |
| 좌표 범위가 입력 크기와 맞는 것 | `Identity` 는 23.7~197.5 로 224 안에 들어옴 |
| 다른 63개는 범위가 다른 것 | `Identity_3` 은 -0.087~0.075 로 좌표로 쓸 수 없음 |
| 확률로 읽히는 것 | `Identity_1` 은 0.9885 로 0~1 안 |
| 두 갈래 점수로 읽히는 것 | `Identity_2` 는 0.1428 로 0.5 아래 |

**후처리가 쓰는 것은 앞의 세 개뿐이다.** `Identity_3` 은 값의 범위가 좌표와 달라 화면 좌표로 쓸 수 없고, 이
배포본의 후처리 코드가 읽지 않는다. 무엇인지 확정할 근거가 원본 코드에 없으므로 **역할은 미확정으로 둔다.**

**63개가 21×3 으로 잘리는 규칙**

```text
[ x0 y0 z0  x1 y1 z1  x2 y2 z2  …  x20 y20 z20 ]
  └ 점 0 ┘  └ 점 1 ┘  └ 점 2 ┘      └ 점 20 ┘
```

`i` 번째 점의 값은 `63` 개 중 `i × 3` 자리에서 세 개다. 세 개씩 끊는 순서를 틀리면 점이 뒤섞여 손 모양이
엉킨다.

**좌표가 어느 공간의 값인지 코드가 판정한다.** 정규화된 0~1 값을 내놓는 모델도 있고 입력 픽셀 좌표를 내놓는
모델도 있어, 후처리가 실제 값을 보고 갈라낸다.

| 값의 범위 | 복원 방법 |
|---|---|
| `0 ≤ x, y ≤ 1` | 원본 크기를 곱한다 |
| 그 밖 | `원본 크기 / 입력 크기` 배율을 곱한다 |

이번 실행의 좌표는 `23.70`~`197.53` 이라 두 번째 갈래다. 배율은 `3009 / 224 = 13.4330` 과
`2592 / 224 = 11.5714` 이며, 손목(0번) 좌표 `(89.97, 197.53)` 은 원본에서 `(1208.6, 2285.7)` 이 된다.

**`z` 는 손목을 0 으로 둔 상대값이다.** 이번 실행에서 손목의 `z` 는 `-0.000002` 로 사실상 0 이고 나머지 점은
`-14.01`~`0` 사이에 놓인다. 카메라로부터의 거리가 아니므로 **미터 같은 단위를 붙일 수 없고**, 같은 손 안에서
어느 점이 더 앞에 있는지 비교하는 데만 쓴다.

#### 21개 점의 번호와 부위

이 모델은 점의 이름 배열을 갖고 있지 않다. 번호와 부위의 대응은 MediaPipe 가 정한 배치를 따르며, 아래 그림이
그 배치다.

![MediaPipe 21 hand landmarks](resources/09_Hand_Landmark_landmarks.png)

출처: [MediaPipe Hand landmarks detection](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker) — Google, CC BY 4.0

그림의 번호와 이 모델 출력의 번호가 같다는 것은 실행 좌표에 번호를 찍어 확인했다. 0번이 손목이고
4·8·12·16·20 이 각각 엄지·검지·중지·약지·소지의 끝이다.

| 번호 | 부위 |
|---|---|
| 0 | 손목 |
| 1~4 | 엄지 (끝이 4) |
| 5~8 | 검지 (끝이 8) |
| 9~12 | 중지 (끝이 12) |
| 13~16 | 약지 (끝이 16) |
| 17~20 | 소지 (끝이 20) |

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 출력이 네 개이고
나열 순서가 런타임 순서와 다른 것을 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "HandLandmarkLite_1.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `confidence_threshold` | 0.3 | 0.5 | 이 값보다 손 확률이 낮으면 결과를 버림 |

파일이 코드 기본값 `0.5` 를 `0.3` 으로 낮춰 둔다. 낮추면 손이 흐릿한 사진에서도 결과가 남고, 손이 없는 사진에서
엉뚱한 좌표가 통과할 여지도 함께 커진다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 설정 파일을 읽는다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_hand.jpg"
CONFIG_PATH = DX_APP / "src" / "python_example" / "hand_landmark" / "handlandmarklite_1" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

print("config", CONFIG)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 좌표를 원본 크기로 되돌리는 데
  원본 크기를 쓴다
- `HandLandmarkResult` — 후처리가 만들어 시각화로 넘기는 결과. 21×3 좌표·신뢰도·손 방향을 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class HandLandmarkResult:
    """21점 좌표와 신뢰도·손 방향. 원본 대응: common/base/i_processor.py"""

    landmarks: np.ndarray = field(default=None)
    confidence: float = 0.0
    handedness: str = "Unknown"


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. **출력 목록의 순서가 뒤 셀에서 `outputs[0]`,
`outputs[1]` 로 꺼내는 순서**이므로 이름과 함께 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output (런타임 순서):")
for index, info in enumerate(output_info):
    print(f"  [{index}] {info['name']:12s} {info['shape']}")

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 RGB 224×224 `uint8` 로 바꾼다. 원본이 크고 정사각이 아니라 축마다 축소 비율이 크게 다르므로, 좌표를
되돌릴 때 그 두 배율을 따로 써야 한다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 224×224
- dtype — `uint8` 유지
- 축소 비율 — 가로 13.43배, 세로 11.57배로 줄어듦

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 화면에 맞게 줄여 표시한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(cv2.resize(image, (900, int(900 * image.shape[0] / image.shape[1]))),
                  f"Input - {IMAGE_PATH.name}", figsize=(8, 7))

#### 전처리 함수 정의

- `handlandmark_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `handlandmark_preprocessor_process` — 이미지 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def handlandmark_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def handlandmark_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 좌표 복원에 쓸 역배율은 `1 / scale_x` 와 `1 / scale_y` 이며 그 값을 함께
출력한다.

In [ ]:
preprocessor_state = handlandmark_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = handlandmark_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.6f} scale_y={ctx.scale_y:.6f}")
print(f"좌표 복원 배율  x {1 / ctx.scale_x:.4f}배  y {1 / ctx.scale_y:.4f}배")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 네 개를 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(f"  [{index}] {output_info[index]['name']:12s} {str(array.shape):10s} "
          f"min={array.min():9.4f} max={array.max():9.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 63개 출력 두 개의 값 범위 — 어느 쪽이 화면 좌표인지 갈라낸다
- 좌표가 0~1 인지 224 이하인지 — 복원 방법이 갈린다
- 두 개의 `[1, 1]` 값 — 확률로 읽히는 쪽과 두 갈래 점수로 읽히는 쪽

In [ ]:
for index in (0, 3):
    array = np.asarray(outputs[index]).flatten()
    print(f"{output_info[index]['name']:12s} min={array.min():9.4f} max={array.max():9.4f} "
          f"→ 화면 좌표로 {'쓸 수 있다' if array.max() > 1.5 else '쓸 수 없다'}")

landmark_raw = np.asarray(outputs[0]).flatten().reshape(21, 3)
print(f"\n좌표 범위  x {landmark_raw[:, 0].min():.2f}~{landmark_raw[:, 0].max():.2f} "
      f"y {landmark_raw[:, 1].min():.2f}~{landmark_raw[:, 1].max():.2f} "
      f"(입력 {INPUT_WIDTH}×{INPUT_HEIGHT} 안)")
print(f"z 범위     {landmark_raw[:, 2].min():.4f} ~ {landmark_raw[:, 2].max():.4f} "
      f"/ 손목(0번) z = {landmark_raw[0, 2]:.6f}")
print(f"\n{output_info[1]['name']} = {float(np.asarray(outputs[1]).flatten()[0]):.4f}  (0~1 → 손 확률)")
print(f"{output_info[2]['name']} = {float(np.asarray(outputs[2]).flatten()[0]):.4f}  "
      f"(0.5 기준 → {'Right' if float(np.asarray(outputs[2]).flatten()[0]) > 0.5 else 'Left'})")

### 4.4 후처리

63개를 21×3 으로 자르고 좌표 공간을 판정해 원본 크기로 되돌린 뒤, 신뢰도가 임계값에 못 미치면 결과를 버린다.
순서는 자르기 → 공간 판정 → 배율 복원 → 신뢰도·손 방향 채우기 → 걸러내기이며 이 순서를 바꾸지 않는다.

**좌표 공간 판정을 점마다 따로 한다.** 한 점이라도 0~1 을 벗어나면 그 점은 픽셀 좌표로 취급되므로, 두 방식이
한 결과 안에서 섞일 수 있는 구조다. 이번 모델은 모든 점이 픽셀 좌표라 한쪽 갈래만 쓰인다.

**결과를 버릴지 정하는 것은 후처리의 마지막 일이다.** 신뢰도가 임계값보다 낮으면 좌표가 있어도 빈 목록을
돌려주며, 그래서 시각화는 손이 없는 사진에서 아무것도 그리지 않는다.

#### 후처리 함수 정의

- `handlandmark_postprocessor_create` — 점 개수·좌표 개수·임계값을 state 에 담음
- `handlandmark_postprocessor_process` — 자르기·복원·걸러내기를 거쳐 Result 생성

In [ ]:
def handlandmark_postprocessor_create(input_width: int, input_height: int, config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {
        "input_width": input_width,
        "input_height": input_height,
        "num_landmarks": 21,
        "coords_per_landmark": 3,
        "confidence_threshold": config.get("confidence_threshold", 0.5),
    }


def handlandmark_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    data = np.asarray(outputs[0]).flatten().astype(np.float32)
    count = min(state["num_landmarks"], data.size // state["coords_per_landmark"])
    landmarks = np.zeros((count, 3), dtype=np.float32)

    for index in range(count):
        offset = index * state["coords_per_landmark"]
        x, y = data[offset], data[offset + 1]
        z = data[offset + 2] if offset + 2 < data.size else 0.0
        if 0.0 <= x <= 1.0 and 0.0 <= y <= 1.0:
            # 정규화 좌표를 내놓는 배포본이면 원본 크기를 곱한다.
            landmarks[index] = [x * ctx.original_width, y * ctx.original_height, z]
        else:
            # 입력 픽셀 좌표이므로 원본/입력 배율을 곱한다.
            landmarks[index] = [x * ctx.original_width / state["input_width"],
                                y * ctx.original_height / state["input_height"], z]

    result = HandLandmarkResult(landmarks=landmarks)
    if len(outputs) > 1:
        result.confidence = float(np.asarray(outputs[1]).flatten()[0])
    else:
        result.confidence = 1.0
    if len(outputs) > 2:
        result.handedness = ("Right" if float(np.asarray(outputs[2]).flatten()[0]) > 0.5
                             else "Left")

    return [result] if result.confidence >= state["confidence_threshold"] else []

#### 후처리 실행

확인할 것:

- 결과 개수 — 신뢰도가 임계값을 넘어 1개가 남는지
- 좌표 shape — `[21, 3]`
- 손가락 끝 좌표 — 원본 크기 안에 들어오는지
- 임계값을 올렸을 때 — 결과가 사라지는지

In [ ]:
postprocessor_state = handlandmark_postprocessor_create(INPUT_WIDTH, INPUT_HEIGHT, CONFIG)
results = handlandmark_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"result count = {len(results)} (threshold={postprocessor_state['confidence_threshold']})")
result = results[0]
print(f"landmarks shape={result.landmarks.shape} confidence={result.confidence:.4f} "
      f"handedness={result.handedness}")

FINGER_TIPS = {0: "손목", 4: "엄지 끝", 8: "검지 끝", 12: "중지 끝", 16: "약지 끝", 20: "소지 끝"}
for index, name in FINGER_TIPS.items():
    x, y, z = result.landmarks[index]
    print(f"  {index:2d} {name:8s} 원본 ({x:7.1f}, {y:7.1f})  z={z:8.4f}")

strict_state = handlandmark_postprocessor_create(INPUT_WIDTH, INPUT_HEIGHT,
                                                 {"confidence_threshold": 0.999})
print(f"\n임계값 0.999 로 올리면 결과 "
      f"{len(handlandmark_postprocessor_process(strict_state, outputs, ctx))}개")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 21점과 연결선을 원본 위에 그린다. 연결 목록이 어느 점과 어느 점을 잇는지 정하며, 손가락별로
색을 달리해 번호 체계를 눈으로 확인할 수 있게 한다. 손목만 점을 크게 그린다. 그림에 넣는 글자는 영문만 쓴다 —
OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),              # 엄지
    (0, 5), (5, 6), (6, 7), (7, 8),              # 검지
    (0, 9), (9, 10), (10, 11), (11, 12),         # 중지
    (0, 13), (13, 14), (14, 15), (15, 16),       # 약지
    (0, 17), (17, 18), (18, 19), (19, 20),       # 소지
    (5, 9), (9, 13), (13, 17),                   # 손바닥
]
FINGER_COLORS = {
    "thumb": (0, 255, 255), "index": (0, 255, 0), "middle": (255, 0, 0),
    "ring": (0, 165, 255), "pinky": (255, 0, 255), "palm": (200, 200, 200),
}


def handlandmark_connection_color(start: int, end: int) -> tuple:
    """연결선이 어느 손가락에 속하는지로 색을 정한다. 원본 대응: HandLandmarkVisualizer._get_connection_color()"""
    for name, span in (("thumb", (1, 4)), ("index", (5, 8)), ("middle", (9, 12)),
                       ("ring", (13, 16)), ("pinky", (17, 20))):
        if span[0] <= end <= span[1] and start != 0 or (start == 0 and span[0] == end):
            return FINGER_COLORS[name]
    return FINGER_COLORS["palm"]


def handlandmark_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    thickness = max(2, int(min(output.shape[:2]) / 300))

    for result in results:
        landmarks = result.landmarks
        if landmarks is None or len(landmarks) == 0:
            continue

        for start, end in HAND_CONNECTIONS:
            point_a = (int(landmarks[start, 0]), int(landmarks[start, 1]))
            point_b = (int(landmarks[end, 0]), int(landmarks[end, 1]))
            cv2.line(output, point_a, point_b, handlandmark_connection_color(start, end),
                     thickness, cv2.LINE_AA)

        for index in range(len(landmarks)):
            point = (int(landmarks[index, 0]), int(landmarks[index, 1]))
            radius = thickness * 3 if index == 0 else thickness * 2
            cv2.circle(output, point, radius, (0, 0, 255), -1)
            cv2.circle(output, point, radius, (255, 255, 255), 1)

        label = f"{result.handedness} ({result.confidence:.2f})"
        cv2.putText(output, label, (int(landmarks[0, 0]), int(landmarks[0, 1]) - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, thickness * 0.5, (0, 255, 0), thickness)

    return output

#### 결과 시각화

21점과 연결선을 원본 위에 그려 표시한다. 원본이 커서 화면에 맞게 줄여 보인다.

In [ ]:
output_image = handlandmark_visualizer_visualize(image, results)
notebook_show_bgr(cv2.resize(output_image, (900, int(900 * image.shape[0] / image.shape[1]))),
                  "21 hand landmarks", figsize=(8, 7))

#### 번호를 찍어 배치 그림과 대조

위 그림은 점의 번호를 보여 주지 않는다. 좌표마다 번호를 적어 표시하면 앞의 배치 그림과 같은 순서인지 직접
확인할 수 있다. 0번이 손목에, 4·8·12·16·20 이 각 손가락 끝에 놓이는지 본다.

In [ ]:
numbered = output_image.copy()
scale = max(0.8, min(numbered.shape[:2]) / 900)
for index in range(len(result.landmarks)):
    x, y = int(result.landmarks[index, 0]), int(result.landmarks[index, 1])
    cv2.putText(numbered, str(index), (x + 12, y - 12), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (255, 255, 0), max(2, int(scale * 2)), cv2.LINE_AA)

notebook_show_bgr(cv2.resize(numbered, (900, int(900 * image.shape[0] / image.shape[1]))),
                  "Landmark index overlay", figsize=(8, 7))